# Daily Truck Distance and Speed Features

This notebook reads the daily GPS position files in `Positions/` and creates one row per truck per day with total route distance and average speed. The output file is `vehicle_daily_distance_speed.csv`.

In [ ]:
import csv
import math
from collections import defaultdict
from pathlib import Path

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


In [ ]:
POSITIONS_DIR = Path("Positions")
OUTPUT_FILE = Path("vehicle_daily_distance_speed.csv")

# Protects the daily distance from rare GPS jumps. Increase or set to None if needed.
MAX_SEGMENT_MILES = 5.0

position_files = sorted(POSITIONS_DIR.glob("2023-*.csv"))
print(f"Found {len(position_files)} daily position files in {POSITIONS_DIR}.")


In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in miles."""
    earth_radius_miles = 3958.7613
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * earth_radius_miles * np.arcsin(np.sqrt(a))


def vehicle_day_features(points, max_segment_miles=MAX_SEGMENT_MILES):
    """Return distance/speed features for one vehicle on one day."""
    points = sorted(points, key=lambda row: row[0])
    lat = np.array([row[1] for row in points], dtype=float)
    lon = np.array([row[2] for row in points], dtype=float)
    speed = np.array([row[3] for row in points], dtype=float)

    valid_position = np.isfinite(lat) & np.isfinite(lon)
    lat = lat[valid_position]
    lon = lon[valid_position]

    if len(lat) < 2:
        total_distance_miles = 0.0
        kept_segments = 0
        dropped_segments = 0
    else:
        segment_miles = haversine_miles(lat[:-1], lon[:-1], lat[1:], lon[1:])
        valid_segments = np.isfinite(segment_miles)
        if max_segment_miles is not None:
            valid_segments &= segment_miles <= max_segment_miles
        total_distance_miles = float(segment_miles[valid_segments].sum())
        kept_segments = int(valid_segments.sum())
        dropped_segments = int(len(segment_miles) - kept_segments)

    valid_speed = speed[np.isfinite(speed)]
    average_speed = float(valid_speed.mean()) if len(valid_speed) else math.nan
    moving_point_count = int((valid_speed > 0).sum()) if len(valid_speed) else 0

    return {
        "total_distance_miles": total_distance_miles,
        "total_distance_km": total_distance_miles * 1.609344,
        "average_speed": average_speed,
        "point_count": len(points),
        "moving_point_count": moving_point_count,
        "kept_segments": kept_segments,
        "dropped_segments": dropped_segments,
    }


In [ ]:
rows = []

for file_index, path in enumerate(position_files, start=1):
    date = path.stem
    points_by_vehicle = defaultdict(list)

    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            try:
                vehicle_id = int(row["VehicleId"])
                latitude = float(row["Latitude"])
                longitude = float(row["Longitude"])
                speed = float(row["Speed"]) if row.get("Speed") not in ("", None) else math.nan
            except (KeyError, TypeError, ValueError):
                continue

            points_by_vehicle[vehicle_id].append((row.get("Timestamp", ""), latitude, longitude, speed))

    for vehicle_id, points in sorted(points_by_vehicle.items()):
        features = vehicle_day_features(points)
        rows.append(
            {
                "date": date,
                "VehicleId": vehicle_id,
                "total_distance_miles": round(features["total_distance_miles"], 3),
                "total_distance_km": round(features["total_distance_km"], 3),
                "average_speed": round(features["average_speed"], 3) if math.isfinite(features["average_speed"]) else "",
                "point_count": features["point_count"],
                "moving_point_count": features["moving_point_count"],
                "kept_segments": features["kept_segments"],
                "dropped_segments": features["dropped_segments"],
            }
        )

    if file_index % 25 == 0 or file_index == len(position_files):
        print(f"Processed {file_index}/{len(position_files)} files; rows so far: {len(rows)}")

fieldnames = [
    "date",
    "VehicleId",
    "total_distance_miles",
    "total_distance_km",
    "average_speed",
    "point_count",
    "moving_point_count",
    "kept_segments",
    "dropped_segments",
]

with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUTPUT_FILE}")


In [ ]:
# Quick sanity checks and a sample plot for one vehicle.
target_vehicle = 689
vehicle_rows = [row for row in rows if row["VehicleId"] == target_vehicle]
print(f"Vehicle {target_vehicle} rows: {len(vehicle_rows)}")
print(vehicle_rows[:5])

if vehicle_rows and plt is not None:
    dates = [row["date"] for row in vehicle_rows]
    distances = [row["total_distance_miles"] for row in vehicle_rows]
    plt.figure(figsize=(12, 5))
    plt.bar(dates, distances)
    plt.xlabel("Date")
    plt.ylabel("Daily distance (miles)")
    plt.title(f"Daily distance of vehicle {target_vehicle}")
    plt.xticks(dates[::30], rotation=45)
    plt.tight_layout()
    plt.show()
elif plt is None:
    print("matplotlib is not installed; skipped plot.")
